In [3]:
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split, Dataset
from torchvision import datasets
from torchvision.transforms import v2
import torchvision.transforms.functional as FF
import torch.nn.functional as F
import random

learning_rate = 1e-3
learning_rate2 = 1e-4
batch_size = 64
epochs_pre = 30
epochs_post = 30

cifar_mean = [0.4914, 0.4822, 0.4465]
cifar_std = [0.2470, 0.2435, 0.2615]

train_transform = v2.Compose([
    v2.ToImage(),
    v2.RandomCrop(32, padding=4),
    v2.RandomHorizontalFlip(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=cifar_mean, std=cifar_std)
])

test_transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=cifar_mean, std=cifar_std)
])

training_data = datasets.CIFAR10(
    root="data", train=True, download=True, transform=train_transform
)

test_data = datasets.CIFAR10(
    root="data", train=False, download=True, transform=test_transform
)

train_dataloader = DataLoader(training_data, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

In [4]:
loss_fn = nn.CrossEntropyLoss()

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using mps device


In [ ]:
class RotationTransform:
    def __call__(self, img):
        rotations = [0, 90, 180, 270]

        angle_idx = random.randint(0, 3)
        angle = rotations[angle_idx]

        img = FF.rotate(img, angle)
        return img, angle_idx
    
class RotationDataset(Dataset):
    def __init__(self):
        transform = v2.Compose([
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean=cifar_mean, std=cifar_std)
        ])
        self.dataset = datasets.CIFAR10(
            root="data", train=True, download=True, transform=transform
        )
        self.rotation_transform = RotationTransform()
    
    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, idx):
        img, _ = self.dataset[idx]
        img, rot_label = self.rotation_transform(img)
        return img, rot_label


rotation_dataset = RotationDataset()

train_rot_dataset, test_rot_dataset = random_split(
    rotation_dataset,
    [49000, 1000]
)

train_rot_dataloader = DataLoader(train_rot_dataset, batch_size=batch_size)
test_rot_dataloader = DataLoader(test_rot_dataset, batch_size=batch_size)

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.fc1 = nn.Linear(64* 8 * 8, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 4, bias=False)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        logits = self.fc3(x)
        return logits
    
model = Net()

In [7]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.to(device)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X = X.to(device)
        y = y.to(device)
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, curr = loss.item(), batch*batch_size + len(X)
            print(f"loss: {loss:>7f} [{curr:>5d}/{size:>5d}]")

def test_loop(dataloader, model, loss_fn):
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct)::>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

for t in range(epochs_pre):
    print(f"Epoch {t+1}\n-----------------------------------")
    train_loop(train_rot_dataloader, model, loss_fn, optimizer)
    test_loop(test_rot_dataloader, model, loss_fn)
print("Done!")

Using mps device


In [7]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.to(device)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X = X.to(device)
        y = y.to(device)
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, curr = loss.item(), batch*batch_size + len(X)
            print(f"loss: {loss:>7f} [{curr:>5d}/{size:>5d}]")

def test_loop(dataloader, model, loss_fn):
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct)::>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [9]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n-----------------------------------")
    train_loop(train_rot_dataloader, model, loss_fn, optimizer)
    test_loop(test_rot_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-----------------------------------
loss: 1.386294 [   64/49000]
loss: 1.386294 [ 6464/49000]
loss: 1.386294 [12864/49000]
loss: 1.386294 [19264/49000]
loss: 1.386294 [25664/49000]
loss: 1.386294 [32064/49000]
loss: 1.386294 [38464/49000]
loss: 1.386294 [44864/49000]
Test Error: 
 Accuracy: 25.5%, Avg loss: 1.386294 

Epoch 2
-----------------------------------
loss: 1.386294 [   64/49000]
loss: 1.386294 [ 6464/49000]
loss: 1.386294 [12864/49000]
loss: 1.386294 [19264/49000]
loss: 1.386294 [25664/49000]
loss: 1.386294 [32064/49000]
loss: 1.386294 [38464/49000]
loss: 1.386294 [44864/49000]
Test Error: 
 Accuracy: 26.3%, Avg loss: 1.386294 

Epoch 3
-----------------------------------
loss: 1.386294 [   64/49000]
loss: 1.386294 [ 6464/49000]
loss: 1.386294 [12864/49000]
loss: 1.386294 [19264/49000]
loss: 1.386294 [25664/49000]
loss: 1.386294 [32064/49000]
loss: 1.386294 [38464/49000]
loss: 1.386294 [44864/49000]
Test Error: 
 Accuracy: 24.1%, Avg loss: 1.386294 

Epoch 4
--------

In [135]:
torch.save(model.state_dict(), "rotnet.pth")

In [25]:
class CifarNet(nn.Module):
    def __init__(self, pretrained_rotnet):
        super().__init__()

        self.conv1 = pretrained_rotnet.conv1
        self.pool = pretrained_rotnet.pool
        self.conv2 = pretrained_rotnet.conv2

        self.fc1 = nn.Linear(16*5*5, 128)
        self.fc2 = nn.Linear(128, 128)
        self.fc3 = nn.Linear(128, 10, bias=False)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        logits = self.fc3(x)
        return logits

In [26]:
rotnet = Net()
rotnet.load_state_dict(torch.load("rotnet.pth"))

<All keys matched successfully>

In [27]:
cmodel = CifarNet(rotnet)

In [28]:
for param in cmodel.conv1.parameters():
    param.requires_grad = False

for param in cmodel.conv2.parameters():
    param.requires_grad = False

In [31]:
loss_fn = nn.CrossEntropyLoss()
learning_rate2 = 1e-4
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, cmodel.parameters()),
    lr=learning_rate2)

epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n-----------------------------------")
    train_loop(train_dataloader, cmodel, loss_fn, optimizer)
    test_loop(test_dataloader, cmodel, loss_fn)
print("Done!")

Epoch 1
-----------------------------------
loss: 1.272067 [   64/50000]
loss: 1.441251 [ 6464/50000]
loss: 1.073567 [12864/50000]
loss: 1.521735 [19264/50000]
loss: 1.299647 [25664/50000]
loss: 1.427033 [32064/50000]
loss: 1.316373 [38464/50000]
loss: 1.105575 [44864/50000]
Test Error: 
 Accuracy: 43.1%, Avg loss: 1.802677 

Epoch 2
-----------------------------------
loss: 1.197354 [   64/50000]
loss: 1.365742 [ 6464/50000]
loss: 0.990189 [12864/50000]
loss: 1.448250 [19264/50000]
loss: 1.209061 [25664/50000]
loss: 1.345900 [32064/50000]
loss: 1.281064 [38464/50000]
loss: 1.115401 [44864/50000]
Test Error: 
 Accuracy: 43.4%, Avg loss: 1.796346 

Epoch 3
-----------------------------------
loss: 1.181841 [   64/50000]
loss: 1.333889 [ 6464/50000]
loss: 0.952096 [12864/50000]
loss: 1.415613 [19264/50000]
loss: 1.180770 [25664/50000]
loss: 1.314175 [32064/50000]
loss: 1.271745 [38464/50000]
loss: 1.121642 [44864/50000]
Test Error: 
 Accuracy: 43.6%, Avg loss: 1.792769 

Epoch 4
--------